In [ ]:
!pip install -q sentence-transformers transformers accelerate

import json
import re
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
LLM_BATCH_SIZE = 280
MAX_INPUT_LENGTH = 320
MAX_NEW_TOKENS = 160

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DEBUG = False
DEBUG_N = 5

with open("joker_task1_retrieval_corpus25_EN.json") as f:
    corpus = json.load(f)

doc_ids = [d["docid"] for d in corpus]
doc_texts = [d["text"] for d in corpus]

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)
tokenizer.padding_side = "left"

llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    device_map="auto",
    torch_dtype=torch.float16
)
llm.eval()

SYSTEM_PROMPT = """You are a humor analysis assistant.

Given a text, decide whether it is a joke.

Output ONLY a valid JSON object enclosed in <json></json> tags.
Do not include any text outside the tags.

The JSON must contain exactly two fields:
- isJoke: boolean
- explanation: a short one-sentence explanation.

Text:
"""

def parse_llm_output(text):
    m = re.search(r"<json>(.*?)</json>", text, re.DOTALL | re.IGNORECASE)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass

    m = re.search(r"\{[\s\S]*?\}", text)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            pass

    return {"isJoke": False, "explanation": ""}

analysis_cache = {}

for i in tqdm(range(0, len(doc_texts), LLM_BATCH_SIZE)):
    batch_texts = doc_texts[i:i + LLM_BATCH_SIZE]
    batch_ids = doc_ids[i:i + LLM_BATCH_SIZE]

    prompts = [
        f"{SYSTEM_PROMPT}{t}\n<json>"
        for t in batch_texts
    ]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    ).to(llm.device)

    with torch.no_grad():
        outputs = llm.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False
        )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for docid, raw in zip(batch_ids, decoded):
        if DEBUG and len(analysis_cache) < DEBUG_N:
            print("\n================ RAW LLM OUTPUT ================")
            print(raw)
            print("================================================")

        parsed = parse_llm_output(raw)

        if DEBUG and len(analysis_cache) < DEBUG_N:
            print("PARSED:", parsed)
            print("================================================\n")

        analysis_cache[str(docid)] = parsed

if DEBUG:
    print("Jokes detected:", sum(1 for v in analysis_cache.values() if v["isJoke"]))

    sample_id = doc_ids[0]
    print("docid type:", type(sample_id))
    print("cache key type:", type(next(iter(analysis_cache.keys()))))
    print("lookup int:", analysis_cache.get(sample_id))
    print("lookup str:", analysis_cache.get(str(sample_id)))

with open("llm_analysis_cache.json", "w") as f:
    json.dump(analysis_cache, f, indent=2)

print("LLM analysis cache saved.")

In [ ]:
!pip install -q sentence-transformers transformers accelerate

import json
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import torch

CONFIG = {
    "embedding_model": "Qwen/Qwen3-Embedding-4B",
    "top_k": 1000,
    "use_explanation": True,
    "filter_mode": "hard",
    "query_mode": "joke"
}

TEAM_ID = "jokeminer"
TASK_ID = "task_1"
METHOD_USED = "_qwen"
RUN_ID = f"{TEAM_ID}_{TASK_ID}_{METHOD_USED}"
MANUAL_FLAG = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open("joker_task1_retrieval_corpus25_EN.json") as f:
    corpus = json.load(f)

with open("joker_task1_retrieval_queries_test25_EN.json") as f:
    queries_test = json.load(f)

with open("llm_analysis_cache.json") as f:
    analysis_cache = json.load(f)

doc_ids = [d["docid"] for d in corpus]
doc_texts = [d["text"] for d in corpus]

embedder = SentenceTransformer(
    CONFIG["embedding_model"],
    device=DEVICE,
    trust_remote_code=True
)

def build_query(query):
    if CONFIG["query_mode"] == "joke":
        return f"Given a query, retrieve relevant jokes related to the query: {query}"
    return query

def build_doc_text(text, analysis):
    if CONFIG["use_explanation"] and analysis.get("explanation"):
        return text + " Explanation: " + analysis["explanation"]
    return text

filtered_doc_ids = []
filtered_texts = []

for docid, text in zip(doc_ids, doc_texts):
    analysis = analysis_cache.get(str(docid), {"isJoke": False, "explanation": ""})

    if not analysis["isJoke"]:
        continue

    filtered_doc_ids.append(docid)
    filtered_texts.append(build_doc_text(text, analysis))

assert len(filtered_texts) > 0, "Hard filter removed entire corpus!"

print("Encoding corpus embeddings with Qwen...")
doc_embeddings = embedder.encode(
    filtered_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64
)

print("Encoding query embeddings...")
query_texts = [build_query(q["query"]) for q in queries_test]

query_embeddings = embedder.encode(
    query_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32
)

predictions = []

for q, q_emb in zip(queries_test, query_embeddings):
    scores = np.dot(doc_embeddings, q_emb)

    idx = np.argsort(scores)[::-1][:CONFIG["top_k"]]
    min_s, max_s = scores[idx].min(), scores[idx].max()

    for rank, i in enumerate(idx, 1):
        predictions.append({
            "run_id": RUN_ID,
            "manual": MANUAL_FLAG,
            "qid": q["qid"],
            "docid": filtered_doc_ids[i],
            "rank": rank,
            "score": float((scores[i] - min_s) / (max_s - min_s + 1e-8))
        })

with open("prediction.json", "w") as f:
    json.dump(predictions, f, indent=2)

!zip submission.zip prediction.json
